In [5]:
# ── Setup: LLM, Tools, and Agent ──

import os, json, base64, shutil, re, requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
from langchain_core.tools import tool

load_dotenv()

OUTPUT_DIR = "output"
DIAGRAMS_DIR = f"{OUTPUT_DIR}/diagrams"
SECTIONS_FILE = f"{OUTPUT_DIR}/sections.json"

llm = ChatOpenAI(model="gpt-5-mini", temperature=0.1)


# ── Tools ──

@tool
def create_mermaid_diagram(mermaid_code: str) -> str:
    """Render Mermaid syntax to a PNG file. Pass raw Mermaid code only (no markdown fences)."""
    os.makedirs(DIAGRAMS_DIR, exist_ok=True)
    encoded = base64.urlsafe_b64encode(mermaid_code.encode()).decode()
    resp = requests.get(f"https://mermaid.ink/img/{encoded}?type=png&bgColor=white", timeout=30)
    if resp.status_code != 200:
        return f"Error: HTTP {resp.status_code}. Check your Mermaid syntax."
    idx = len([f for f in os.listdir(DIAGRAMS_DIR) if f.endswith(".png")]) + 1
    path = f"{DIAGRAMS_DIR}/diagram_{idx}.png"
    with open(path, "wb") as f:
        f.write(resp.content)
    return f"Diagram saved to {path}"


@tool
def save_research_section(input_str: str) -> str:
    """Save a section for the PDF. Input: JSON with "title" and "content" keys."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    title, content = _parse_section(input_str)
    sections = json.loads(open(SECTIONS_FILE).read()) if os.path.exists(SECTIONS_FILE) else []
    sections.append({"title": title, "content": content})
    with open(SECTIONS_FILE, "w") as f:
        json.dump(sections, f)
    return f"Saved '{title}' ({len(content)} chars). Total: {len(sections)}"


def _parse_section(raw: str) -> tuple[str, str]:
    """Parse JSON from agent input, handling quirks like extra quotes."""
    raw = raw.strip()
    if (raw[0], raw[-1]) in [("'", "'"), ('"', '"')]:
        raw = raw[1:-1]
    try:
        data = json.loads(raw)
        return data["title"], data["content"]
    except (json.JSONDecodeError, KeyError):
        pass
    try:
        data = json.loads(json.loads(f'"{raw}"'))
        return data["title"], data["content"]
    except Exception:
        lines = raw.split("\n", 1)
        return lines[0][:100], lines[1] if len(lines) > 1 else raw


tools = [DuckDuckGoSearchRun(), WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)), create_mermaid_diagram, save_research_section]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['duckduckgo_search', 'wikipedia', 'create_mermaid_diagram', 'save_research_section']


In [6]:
# ── Run Agent ──

# Clean previous output
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(DIAGRAMS_DIR, exist_ok=True)

# Build agent
agent = create_react_agent(llm, tools, hub.pull("hwchase17/react"))
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True, max_iterations=25)

# Get user input and wrap with system instructions
user_topic = input("Enter your research topic: ")

SYSTEM_PROMPT = f"""Research the following topic: {user_topic}

You MUST complete ALL of these steps:

1. RESEARCH using duckduckgo_search and wikipedia to gather detailed information.

2. SAVE each finding using save_research_section with JSON input: {{"title": "...", "content": "..."}}

3. CREATE at least 2-3 Mermaid diagrams using create_mermaid_diagram to visualize key concepts, relationships, or processes from your research. Use raw Mermaid syntax (no markdown fences).

Complete ALL steps before giving your final answer."""

print(f"\nResearching: {user_topic}\n" + "-" * 60)
response = executor.invoke({"input": SYSTEM_PROMPT})
print("-" * 60 + f"\n\nFinal Output:\n{response['output']}")


Researching: 
------------------------------------------------------------


> Entering new AgentExecutor chain...


BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'stop' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'stop', 'code': 'unsupported_parameter'}}

In [ ]:
# ── Generate PDF Report ──

from fpdf import FPDF
from glob import glob
from PIL import Image

PAGE_W, PAGE_H, MARGIN = 210, 297, 20
MAX_IMG_H = PAGE_H - MARGIN - 22 - 24  # usable height minus header/label/gaps


class ResearchPDF(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.set_text_color(100, 100, 100)
        self.cell(0, 8, "Research Report", align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f"Page {self.page_no()}/{{nb}}", align="C")

    def write_markdown(self, text):
        """Render text, converting **bold** markers to actual bold font."""
        self.set_font("Helvetica", "", 11)
        self.set_text_color(50, 50, 50)
        for part in re.split(r"(\*\*.*?\*\*)", text):
            if part.startswith("**") and part.endswith("**"):
                self.set_font("Helvetica", "B", 11)
                self.write(6, part[2:-2])
                self.set_font("Helvetica", "", 11)
            else:
                self.write(6, part)

    def add_image_fit(self, img_path, label):
        """Add image scaled to fit the page, with automatic page break."""
        with Image.open(img_path) as img:
            w_px, h_px = img.size
        img_w, img_h = 180, (180 / w_px) * h_px
        if img_h > MAX_IMG_H:  # scale down tall images
            img_w = (MAX_IMG_H / h_px) * w_px
            img_h = MAX_IMG_H
        if 28 + img_h > PAGE_H - MARGIN - self.get_y():
            self.add_page()
        self.set_font("Helvetica", "B", 13)
        self.set_text_color(30, 30, 30)
        self.cell(0, 10, label, new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
        self.image(img_path, x=(PAGE_W - img_w) / 2, w=img_w, h=img_h)
        self.ln(img_h + 10)


# Build PDF
pdf = ResearchPDF()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=MARGIN)

# Title page
pdf.add_page()
pdf.ln(60)
pdf.set_font("Helvetica", "B", 28)
pdf.set_text_color(30, 30, 30)
pdf.cell(0, 15, user_topic[:50], align="C", new_x="LMARGIN", new_y="NEXT")
pdf.ln(10)
pdf.set_font("Helvetica", "", 14)
pdf.set_text_color(80, 80, 80)
pdf.cell(0, 10, "A Visual Research Report", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "I", 11)
pdf.cell(0, 10, "Generated by an AI Agent using LangChain + Mermaid", align="C", new_x="LMARGIN", new_y="NEXT")

# Research sections
sections = json.loads(open(SECTIONS_FILE).read()) if os.path.exists(SECTIONS_FILE) else []
for i, sec in enumerate(sections):
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 18)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(0, 12, f"{i + 1}. {sec['title']}", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)
    pdf.write_markdown(sec["content"])
    pdf.ln(8)

# Diagrams
diagrams = sorted(glob(f"{DIAGRAMS_DIR}/diagram_*.png"))
if diagrams:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 22)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(0, 14, "Visual Diagrams", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)
    for idx, path in enumerate(diagrams):
        pdf.add_image_fit(path, f"Diagram {idx + 1}")

# Save
pdf_path = f"{OUTPUT_DIR}/Research_Report.pdf"
pdf.output(pdf_path)
print(f"PDF saved: {pdf_path} ({len(sections)} sections, {len(diagrams)} diagrams)")